# 12 - Prepare Azure tools

Install the `containerapp` CLI extension if missing and register `Microsoft.App` if needed. These cells change CLI setup and provider registration. They do not create paid resources, change the project kernel, or print account IDs.


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())

preflight = json.loads((ROOT / "reports/azure-preflight.json").read_text())
assert preflight["subscription_enabled"] and not preflight["checks_failed"], "Finish 11 first."
print("Ready to prepare CLI tools.")


Ready to prepare CLI tools.


## Set up

The CLI now finds containerapp 1.3.0b5. After the timeout fix, run only the final status cell first. Do not reinstall an existing extension.

For a new machine, this cell installs the pinned extension if missing and starts provider registration if needed. Existing setup is kept. Installation has a 10-minute limit. Stop on errors.


In [2]:
def check_tools(status_only=False):
    args = [sys.executable, "-m", "cloudlayer.setup_tools"]
    if status_only:
        args.append("--status-only")
    result = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Setup script failed. Read the traceback above.")
    report = json.loads(result.stdout)
    print(json.dumps(report, indent=2))
    assert not report["errors"] and not report["checks_failed"], "Stop and send the report."
    return report

report = check_tools()


{
  "checked_at": "2026-10-11T13:22:32.233366+00:00",
  "changes": [
    "register_Microsoft.App"
  ],
  "errors": [
    {
      "step": "install_containerapp_extension",
      "code": "CLIError"
    }
  ],
  "checks_failed": [],
  "resources_created": false,
  "status_only": false,
  "setup_ready": false,
  "extensions": [
    "ml"
  ],
  "provider_state": "Registering",
  "next": "Read status again if Registering; stop on errors."
}


AssertionError: Stop and send the report.

## Read status

Run this cell now, then save. It reads extension metadata and checks Azure state without loading unrelated ML commands. It does not install packages or change provider registration.

Expect no errors and containerapp 1.3.0b5 in extension_versions. If the provider is Registering, continue to 13 and repeat only this cell later. When Registered, setup_ready becomes true. Output: reports/azure-tools.json.

[Azure setup guide](https://learn.microsoft.com/en-us/azure/container-apps/containerapp-up)


In [ ]:
report = check_tools(status_only=True)
print("Setup ready:", report["setup_ready"])
